<a href="https://colab.research.google.com/github/mishra-bytes/Face-Orientation-Detector/blob/main/Face_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics mediapipe seaborn pandas


In [ ]:

import os
import cv2
import time
import random
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image

# Force clean old files to ensure fresh downloads
!rm -f yunet.onnx deploy.prototxt res10_300x300_ssd_iter_140000.caffemodel face_landmarker.task

print(">>> Downloading fresh model files...")
!wget -q -O yunet.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -q https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
!wget -q https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel


IMAGE_DIR = "/content/drive/MyDrive/InnerGize_Face/data/test"
NUM_VISUAL_SAMPLES = 3
BENCHMARK_RUNS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GENERAL_EXPAND = 10
DNN_PADDING_RATIO = 0.05

print(f"Environment Ready. Device: {DEVICE}")


face_landmarker = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="face_landmarker.task"),
        num_faces=1
    )
)

# --- B. OpenCV Models ---
dnn_net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "res10_300x300_ssd_iter_140000.caffemodel")
haar = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
yunet = cv2.FaceDetectorYN.create(
    model="yunet.onnx", config="", input_size=(640, 640),
    score_threshold=0.5, nms_threshold=0.3, top_k=5000
)

# --- C. YOLO Models ---
yolo_detect = YOLO("yolov8n.pt")
yolo_pose   = YOLO("yolov8n-pose.pt")
yolo_cls    = YOLO("yolov8n-cls.pt")

# --- D. Torchvision Segmentation Models (ALL VARIANTS) ---
def load_seg(model_fn, weights):
    return model_fn(weights=weights).to(DEVICE).eval()

SEG_MODELS = {
    "DeepLabV3-ResNet50": load_seg(models.segmentation.deeplabv3_resnet50, models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT),
    "FCN-ResNet50": load_seg(models.segmentation.fcn_resnet50, models.segmentation.FCN_ResNet50_Weights.DEFAULT),
    "DeepLabV3-MobileNetV3": load_seg(models.segmentation.deeplabv3_mobilenet_v3_large, models.segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT),
    "LRASPP-MobileNetV3": load_seg(models.segmentation.lraspp_mobilenet_v3_large, models.segmentation.LRASPP_MobileNet_V3_Large_Weights.DEFAULT),
    "DeepLabV3-ResNet101": load_seg(models.segmentation.deeplabv3_resnet101, models.segmentation.DeepLabV3_ResNet101_Weights.DEFAULT),
    "FCN-ResNet101": load_seg(models.segmentation.fcn_resnet101, models.segmentation.FCN_ResNet101_Weights.DEFAULT),
}

torch_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"Loaded {7 + len(SEG_MODELS)} Models Total.")


all_images = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
if not all_images: raise FileNotFoundError(f"No images in {IMAGE_DIR}")
samples = random.sample(all_images, min(NUM_VISUAL_SAMPLES, len(all_images)))

print("\n>>> STARTING VISUALIZATION (Generating results for ALL models)...")

for idx, path in enumerate(samples, 1):
    print(f"\n[{idx}/{len(samples)}] Processing: {os.path.basename(path)}")
    img_bgr = cv2.imread(path)
    if img_bgr is None: continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    # Store all results here
    results = {"Original": img_rgb}

    # 1. MediaPipe
    mp_image = mp.Image(mp.ImageFormat.SRGB, img_rgb)
    lm_result = face_landmarker.detect(mp_image)
    mask_lm = np.zeros((h, w), dtype=np.uint8)
    if lm_result.face_landmarks:
        pts = np.array([[int(l.x * w), int(l.y * h)] for l in lm_result.face_landmarks[0]], dtype=np.int32)
        cv2.fillConvexPoly(mask_lm, cv2.convexHull(pts), 255)
    results["MediaPipe"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_lm)

    # 2. SSD
    blob = cv2.dnn.blobFromImage(img_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0))
    dnn_net.setInput(blob)
    detections = dnn_net.forward()
    mask_dnn = np.zeros((h, w), dtype=np.uint8)
    if detections.shape[2] > 0:
        for i in range(detections.shape[2]):
            if detections[0, 0, i, 2] > 0.5:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                x1, y1, x2, y2 = box.astype(int)
                bw, bh = x2-x1, y2-y1
                pad_w, pad_h = int(bw*DNN_PADDING_RATIO), int(bh*DNN_PADDING_RATIO)
                mask_dnn[max(0, y1-pad_h):min(h, y2+pad_h), max(0, x1-pad_w):min(w, x2+pad_w)] = 255
                break
    results["DNN SSD"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_dnn)

    # 3. Haar
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = haar.detectMultiScale(gray, 1.1, 5)
    mask_haar = np.zeros((h, w), dtype=np.uint8)
    if len(faces):
        x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
        mask_haar[y:y+fh, x:x+fw] = 255
    results["Haar"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_haar)

    # 4. YuNet
    yunet.setInputSize((w, h))
    _, faces_yu = yunet.detect(img_bgr)
    mask_yu = np.zeros((h, w), dtype=np.uint8)
    if faces_yu is not None and len(faces_yu) > 0:
        face = faces_yu[0]
        x, y, bw, bh = face[0:4].astype(int)
        mask_yu[y:y+bh, x:x+bw] = 255
    results["YuNet"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_yu)

    # 5. YOLO Detect
    r_det = yolo_detect(img_rgb, conf=0.4, verbose=False)[0]
    mask_det = np.zeros((h, w), dtype=np.uint8)
    if r_det.boxes and len(r_det.boxes) > 0:
        b = r_det.boxes.xyxy.cpu().numpy()[0].astype(int)
        mask_det[b[1]:b[3], b[0]:b[2]] = 255
    results["YOLO Detect"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_det)

    # 6. YOLO Pose
    r_pose = yolo_pose(img_rgb, conf=0.4, verbose=False)[0]
    mask_pose = np.zeros((h, w), dtype=np.uint8)
    if r_pose.keypoints is not None and len(r_pose.keypoints.xy) > 0:
        kps = r_pose.keypoints.xy.cpu().numpy()[0].astype(int)
        cv2.fillConvexPoly(mask_pose, cv2.convexHull(kps), 255)
    results["YOLO Pose"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_pose)

    # 7. ALL Segmentation Models
    input_tensor = torch_preprocess(img_rgb).unsqueeze(0).to(DEVICE)
    for name, model in SEG_MODELS.items():
        with torch.no_grad(): out = model(input_tensor)["out"]
        pred = out[0].argmax(0).cpu().numpy()
        mask_seg = cv2.resize(((pred == 15).astype(np.uint8) * 255), (w, h), interpolation=cv2.INTER_NEAREST)
        results[name] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_seg)

    # Plotting Grid (4 columns)
    cols = 4
    rows = (len(results) + cols - 1) // cols
    plt.figure(figsize=(24, 6 * rows))
    for i, (name, img) in enumerate(results.items(), 1):
        plt.subplot(rows, cols, i)
        plt.title(name, fontsize=11, fontweight='bold')
        plt.imshow(img)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


print("\n>>> STARTING BENCHMARKING (Collecting Data)...")

bench_img_bgr = cv2.imread(samples[0])
bench_img_rgb = cv2.cvtColor(bench_img_bgr, cv2.COLOR_BGR2RGB)
h, w, _ = bench_img_rgb.shape

# Inputs
mp_input = mp.Image(mp.ImageFormat.SRGB, bench_img_rgb)
blob_input = cv2.dnn.blobFromImage(bench_img_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0))
torch_input = torch_preprocess(bench_img_rgb).unsqueeze(0).to(DEVICE)

data = []

# Warmup
with torch.no_grad():
    for _ in range(2):
        for m in SEG_MODELS.values(): _ = m(torch_input)
        yolo_detect(bench_img_rgb, verbose=False)

# Loop
for i in range(BENCHMARK_RUNS):
    # Detectors
    t0=time.perf_counter(); face_landmarker.detect(mp_input); data.append({"Model":"MediaPipe","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    t0=time.perf_counter(); dnn_net.setInput(blob_input); dnn_net.forward(); data.append({"Model":"DNN SSD","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    gray_b=cv2.cvtColor(bench_img_bgr,cv2.COLOR_BGR2GRAY)
    t0=time.perf_counter(); haar.detectMultiScale(gray_b,1.1,5); data.append({"Model":"Haar","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    yunet.setInputSize((w, h))
    t0=time.perf_counter(); yunet.detect(bench_img_bgr); data.append({"Model":"YuNet","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    t0=time.perf_counter(); yolo_detect(bench_img_rgb,verbose=False); data.append({"Model":"YOLO Detect","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    t0=time.perf_counter(); yolo_pose(bench_img_rgb,verbose=False); data.append({"Model":"YOLO Pose","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    t0=time.perf_counter(); yolo_cls(bench_img_rgb,verbose=False); data.append({"Model":"YOLO Class","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    # Segmentation (All Variants)
    if DEVICE=="cuda": torch.cuda.synchronize()
    for name, model in SEG_MODELS.items():
        t0=time.perf_counter()
        with torch.no_grad(): _=model(torch_input)["out"]
        if DEVICE=="cuda": torch.cuda.synchronize()
        data.append({"Model":name, "Run":i, "Latency_ms":(time.perf_counter()-t0)*1000})

df = pd.DataFrame(data)


summary = df.groupby('Model')['Latency_ms'].agg(['mean','median','std','min','max']).sort_values('mean')
summary['FPS'] = 1000.0 / summary['mean']
summary['P95'] = df.groupby('Model')['Latency_ms'].quantile(0.95)
summary = summary.round(2)

print("\n" + "="*100)
print(f"{'FULL MODEL PERFORMANCE REPORT':^100}")
print("="*100)
print(summary[['mean','FPS','std','median','P95']].to_string())
print("="*100 + "\n")

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
fig = plt.figure(figsize=(24, 22))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.2])

# 1. Bar Plot (FPS)
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x="FPS", y=summary.index, data=summary, palette="viridis", ax=ax1, orient='h')
ax1.set_title("Throughput (FPS) - Higher is Better", fontweight='bold')
ax1.bar_label(ax1.containers[0], fmt='%.1f')

# 2. Heatmap
ax2 = fig.add_subplot(gs[0, 1])
hm_data = summary[['mean', 'median', 'P95', 'std']].copy()
sns.heatmap(hm_data, annot=True, fmt=".1f", cmap="YlGnBu", cbar=False, ax=ax2)
ax2.set_title("Latency Scorecard (ms)", fontweight='bold')

# 3. Box Plot (Log Scale for visibility of fast vs slow models)
ax3 = fig.add_subplot(gs[1, :])
sns.boxplot(x="Model", y="Latency_ms", data=df, hue="Model", palette="Spectral", ax=ax3, showfliers=False, legend=False)
sns.stripplot(x="Model", y="Latency_ms", data=df, color=".2", size=3, alpha=0.5, ax=ax3)
ax3.set_yscale("log")
ax3.set_title("Latency Distribution (Log Scale)", fontweight='bold')
ax3.tick_params(axis='x', rotation=30)


ax4 = fig.add_subplot(gs[2, :])

# Define the exact 13 colors
custom_colors = [
    "blue", "orange", "green", "red", "purple", "cyan", "magenta",
    "brown", "teal", "darkgray", "black", "yellow", "pink"
]

sns.lineplot(
    x="Run",
    y="Latency_ms",
    hue="Model",
    data=df,
    palette=custom_colors,
    marker="o",
    linewidth=2,
    ax=ax4
)

ax4.set_title("Inference Stability (Run-by-Run)", fontweight='bold')
ax4.set_ylabel("Latency (ms)")
ax4.legend(bbox_to_anchor=(1.0, 1.0), loc='upper left', ncol=1, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import time
import random
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image

# Extra plotting util
!pip -q install -U statsmodels
from statsmodels.distributions.empirical_distribution import ECDF

# Clean old files
!rm -f yunet.onnx deploy.prototxt res10_300x300_ssd_iter_140000.caffemodel face_landmarker.task

print(">>> Downloading fresh model files...")
!wget -q -O yunet.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -q -O deploy.prototxt https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
!wget -q -O res10_300x300_ssd_iter_140000.caffemodel https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel

IMAGE_DIR = "/content/drive/MyDrive/InnerGize_Face/data/test"
NUM_VISUAL_SAMPLES = 5
BENCHMARK_RUNS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DNN_PADDING_RATIO = 0.05

# YOLO smart-crop output size (only for the added pipeline)
YOLO_SMART_IMG_SIZE = 512

print(f"Environment Ready. Device: {DEVICE}")

# ---------------- A) MediaPipe ----------------
face_landmarker = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="face_landmarker.task"),
        num_faces=1
    )
)

# ---------------- B) OpenCV ----------------
dnn_net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "res10_300x300_ssd_iter_140000.caffemodel")
haar = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
yunet = cv2.FaceDetectorYN.create(
    model="yunet.onnx", config="", input_size=(640, 640),
    score_threshold=0.5, nms_threshold=0.3, top_k=5000
)

# ---------------- C) YOLO baseline ----------------
yolo_detect = YOLO("yolov8n.pt")
yolo_pose   = YOLO("yolov8n-pose.pt")
yolo_cls    = YOLO("yolov8n-cls.pt")

# Added YOLO11 (guarded)
yolo11_detect = yolo11_pose = yolo11_cls = None
try:
    yolo11_detect = YOLO("yolo11n.pt")
    yolo11_pose   = YOLO("yolo11n-pose.pt")
    yolo11_cls    = YOLO("yolo11n-cls.pt")
except Exception as e:
    print("YOLO11 init skipped:", repr(e))

# Added YOLO segmentation for the smart pipeline (guarded)
yolo_seg_v8 = yolo_seg_11 = None
try:
    yolo_seg_v8 = YOLO("yolov8n-seg.pt")
except Exception as e:
    print("YOLOv8 Seg init skipped:", repr(e))

try:
    yolo_seg_11 = YOLO("yolo11n-seg.pt")
except Exception as e:
    print("YOLO11 Seg init skipped:", repr(e))

# ---------------- D) Torchvision segmentation ----------------
def load_seg(model_fn, weights):
    return model_fn(weights=weights).to(DEVICE).eval()

SEG_MODELS = {
    "DeepLabV3-ResNet50": load_seg(models.segmentation.deeplabv3_resnet50, models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT),
    "FCN-ResNet50": load_seg(models.segmentation.fcn_resnet50, models.segmentation.FCN_ResNet50_Weights.DEFAULT),
    "DeepLabV3-MobileNetV3": load_seg(models.segmentation.deeplabv3_mobilenet_v3_large, models.segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT),
    "LRASPP-MobileNetV3": load_seg(models.segmentation.lraspp_mobilenet_v3_large, models.segmentation.LRASPP_MobileNet_V3_Large_Weights.DEFAULT),
    "DeepLabV3-ResNet101": load_seg(models.segmentation.deeplabv3_resnet101, models.segmentation.DeepLabV3_ResNet101_Weights.DEFAULT),
    "FCN-ResNet101": load_seg(models.segmentation.fcn_resnet101, models.segmentation.FCN_ResNet101_Weights.DEFAULT),
}

torch_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print(f"Loaded baseline models + added slots. Seg slots: {len(SEG_MODELS)}")

# ---------------- Color system (one mapping for all plots) ----------------
def build_color_map(model_names):
    base = list(plt.get_cmap("tab20").colors) + list(plt.get_cmap("tab20b").colors) + list(plt.get_cmap("tab20c").colors)
    if len(base) < len(model_names):
        base = base * ((len(model_names) // len(base)) + 1)
    return {name: base[i] for i, name in enumerate(model_names)}

# ---------------- Added: YOLO smart pipeline (as given) ----------------
def yolo_smart_preprocess(yolo_seg_model, img_rgb, img_size=224):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    h, w = img_rgb.shape[:2]

    person_mask = np.zeros((h, w), dtype=np.uint8)

    if yolo_seg_model is not None:
        results = yolo_seg_model(cv2.resize(img_rgb, (640, 640)), verbose=False)
        if hasattr(results[0], "masks") and results[0].masks is not None:
            if results[0].masks.data is not None and len(results[0].masks.data) > 0:
                areas = results[0].boxes.xywh[:, 2] * results[0].boxes.xywh[:, 3]
                idx = int(torch.argmax(areas).item())
                m = results[0].masks.data[idx].detach().cpu().numpy()
                person_mask = (cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)

    if not np.any(person_mask):
        cv2.rectangle(person_mask, (w // 4, h // 4), (3 * w // 4, 3 * h // 4), 1, -1)

    isolated = np.where(np.stack((person_mask,) * 3, axis=-1) > 0, img_rgb, 0)
    gray_img = cv2.cvtColor(isolated, cv2.COLOR_RGB2GRAY)

    y_head = int(np.argmax(np.any(person_mask, axis=1))) if np.any(person_mask) else 0
    roi_h = max(h // 3, 300)
    dim = max(int(roi_h * 1.8), 300)

    denom = (np.sum(person_mask) + 1e-6)
    center_x = int(np.dot(np.arange(w), np.sum(person_mask, axis=0)) / denom)

    half = dim // 2
    x1 = center_x - half
    y1 = y_head + (roi_h // 2) - half

    p_l, p_t = max(0, -x1), max(0, -y1)
    p_r, p_b = max(0, (x1 + dim) - w), max(0, (y1 + dim) - h)

    padded_gray = cv2.copyMakeBorder(gray_img, p_t, p_b, p_l, p_r, cv2.BORDER_CONSTANT, value=0)
    padded_rgb  = cv2.copyMakeBorder(img_rgb,  p_t, p_b, p_l, p_r, cv2.BORDER_CONSTANT, value=0)

    cx1, cy1 = x1 + p_l, y1 + p_t
    crop_gray = padded_gray[cy1:cy1 + dim, cx1:cx1 + dim]
    crop_gray = cv2.resize(crop_gray, (img_size, img_size), interpolation=cv2.INTER_AREA)

    norm_img = crop_gray.astype(np.float32) / 255.0
    norm_img = (norm_img - 0.5) / 0.5
    blob = np.expand_dims(np.expand_dims(norm_img, axis=0), axis=0)

    meta = {
        "person_mask": person_mask,
        "isolated_rgb": isolated,
        "padded_rgb": padded_rgb,
        "crop_coords": (int(cx1), int(cy1), int(cx1 + dim), int(cy1 + dim)),
        "orig_size": (int(h), int(w)),
        "pad_info": (int(p_t), int(p_l)),
        "dim": int(dim),
    }
    return blob.astype(np.float32), meta

# ---------------- Load images ----------------
all_images = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
if not all_images:
    raise FileNotFoundError(f"No images in {IMAGE_DIR}")
samples = random.sample(all_images, min(NUM_VISUAL_SAMPLES, len(all_images)))

print("\n>>> STARTING VISUALIZATION (Generating results for ALL models)...")

for idx, path in enumerate(samples, 1):
    print(f"\n[{idx}/{len(samples)}] Processing: {os.path.basename(path)}")
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    results = {"Original": img_rgb}

    # 1. MediaPipe landmarker
    mp_image = mp.Image(mp.ImageFormat.SRGB, img_rgb)
    lm_result = face_landmarker.detect(mp_image)
    mask_lm = np.zeros((h, w), dtype=np.uint8)
    if lm_result.face_landmarks:
        pts = np.array([[int(l.x * w), int(l.y * h)] for l in lm_result.face_landmarks[0]], dtype=np.int32)
        cv2.fillConvexPoly(mask_lm, cv2.convexHull(pts), 255)
    results["MediaPipe"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_lm)

    # 2. SSD
    blob = cv2.dnn.blobFromImage(img_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0))
    dnn_net.setInput(blob)
    detections = dnn_net.forward()
    mask_dnn = np.zeros((h, w), dtype=np.uint8)
    if detections.shape[2] > 0:
        for i in range(detections.shape[2]):
            if detections[0, 0, i, 2] > 0.5:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                x1, y1, x2, y2 = box.astype(int)
                bw, bh = x2 - x1, y2 - y1
                pad_w, pad_h = int(bw * DNN_PADDING_RATIO), int(bh * DNN_PADDING_RATIO)
                mask_dnn[max(0, y1 - pad_h):min(h, y2 + pad_h), max(0, x1 - pad_w):min(w, x2 + pad_w)] = 255
                break
    results["DNN SSD"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_dnn)

    # 3. Haar
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = haar.detectMultiScale(gray, 1.1, 5)
    mask_haar = np.zeros((h, w), dtype=np.uint8)
    if len(faces):
        x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
        mask_haar[y:y + fh, x:x + fw] = 255
    results["Haar"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_haar)

    # 4. YuNet
    yunet.setInputSize((w, h))
    _, faces_yu = yunet.detect(img_bgr)
    mask_yu = np.zeros((h, w), dtype=np.uint8)
    if faces_yu is not None and len(faces_yu) > 0:
        face = faces_yu[0]
        x, y, bw, bh = face[0:4].astype(int)
        mask_yu[y:y + bh, x:x + bw] = 255
    results["YuNet"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_yu)

    # 5. YOLOv8 Detect
    r_det = yolo_detect(img_rgb, conf=0.4, verbose=False)[0]
    mask_det = np.zeros((h, w), dtype=np.uint8)
    if r_det.boxes and len(r_det.boxes) > 0:
        b = r_det.boxes.xyxy.cpu().numpy()[0].astype(int)
        mask_det[b[1]:b[3], b[0]:b[2]] = 255
    results["YOLO Detect"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_det)

    # 6. YOLOv8 Pose
    r_pose = yolo_pose(img_rgb, conf=0.4, verbose=False)[0]
    mask_pose = np.zeros((h, w), dtype=np.uint8)
    if r_pose.keypoints is not None and len(r_pose.keypoints.xy) > 0:
        kps = r_pose.keypoints.xy.cpu().numpy()[0].astype(int)
        if len(kps) >= 3:
            cv2.fillConvexPoly(mask_pose, cv2.convexHull(kps), 255)
    results["YOLO Pose"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_pose)

    # 6b. YOLO11 Detect
    if yolo11_detect is not None:
        r11 = yolo11_detect(img_rgb, conf=0.4, verbose=False)[0]
        mask11 = np.zeros((h, w), dtype=np.uint8)
        if r11.boxes and len(r11.boxes) > 0:
            b = r11.boxes.xyxy.cpu().numpy()[0].astype(int)
            mask11[b[1]:b[3], b[0]:b[2]] = 255
        results["YOLO11 Detect"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask11)

    # 6c. YOLO11 Pose
    if yolo11_pose is not None:
        r11p = yolo11_pose(img_rgb, conf=0.4, verbose=False)[0]
        mask11p = np.zeros((h, w), dtype=np.uint8)
        if r11p.keypoints is not None and len(r11p.keypoints.xy) > 0:
            kps = r11p.keypoints.xy.cpu().numpy()[0].astype(int)
            if len(kps) >= 3:
                cv2.fillConvexPoly(mask11p, cv2.convexHull(kps), 255)
        results["YOLO11 Pose"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask11p)

    # 6d. Added YOLO smart pipeline (seg mask + crop)
    if yolo_seg_v8 is not None:
        _, meta = yolo_smart_preprocess(yolo_seg_v8, img_rgb, img_size=YOLO_SMART_IMG_SIZE)
        results["YOLOv8-Seg Smart Isolated"] = meta["isolated_rgb"]
    if yolo_seg_11 is not None:
        _, meta = yolo_smart_preprocess(yolo_seg_11, img_rgb, img_size=YOLO_SMART_IMG_SIZE)
        results["YOLO11-Seg Smart Isolated"] = meta["isolated_rgb"]

    # 7. Torchvision segmentation models
    input_tensor = torch_preprocess(img_rgb).unsqueeze(0).to(DEVICE)
    for name, model in SEG_MODELS.items():
        with torch.no_grad():
            out = model(input_tensor)["out"]
        pred = out[0].argmax(0).cpu().numpy()
        mask_seg = cv2.resize(((pred == 15).astype(np.uint8) * 255), (w, h), interpolation=cv2.INTER_NEAREST)
        results[name] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_seg)

    # Plot grid
    cols = 4
    rows = (len(results) + cols - 1) // cols
    plt.figure(figsize=(24, 6 * rows))
    for i, (name, img) in enumerate(results.items(), 1):
        plt.subplot(rows, cols, i)
        plt.title(name, fontsize=11, fontweight='bold')
        plt.imshow(img)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

print("\n>>> STARTING BENCHMARKING (Collecting Data)...")

bench_img_bgr = cv2.imread(samples[0])
bench_img_rgb = cv2.cvtColor(bench_img_bgr, cv2.COLOR_BGR2RGB)
h, w, _ = bench_img_rgb.shape

# Inputs
mp_input = mp.Image(mp.ImageFormat.SRGB, bench_img_rgb)
blob_input = cv2.dnn.blobFromImage(bench_img_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0))
torch_input = torch_preprocess(bench_img_rgb).unsqueeze(0).to(DEVICE)

data = []

# Warmup
with torch.no_grad():
    for _ in range(2):
        for m in SEG_MODELS.values():
            _ = m(torch_input)["out"]
        _ = yolo_detect(bench_img_rgb, verbose=False)
        if yolo11_detect is not None:
            _ = yolo11_detect(bench_img_rgb, verbose=False)
        if yolo_seg_v8 is not None:
            _ = yolo_smart_preprocess(yolo_seg_v8, bench_img_rgb, img_size=YOLO_SMART_IMG_SIZE)
        if yolo_seg_11 is not None:
            _ = yolo_smart_preprocess(yolo_seg_11, bench_img_rgb, img_size=YOLO_SMART_IMG_SIZE)

# Loop
for i in range(BENCHMARK_RUNS):
    # Detectors
    t0=time.perf_counter(); face_landmarker.detect(mp_input); data.append({"Model":"MediaPipe","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    t0=time.perf_counter(); dnn_net.setInput(blob_input); dnn_net.forward(); data.append({"Model":"DNN SSD","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    gray_b=cv2.cvtColor(bench_img_bgr,cv2.COLOR_BGR2GRAY)
    t0=time.perf_counter(); haar.detectMultiScale(gray_b,1.1,5); data.append({"Model":"Haar","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    yunet.setInputSize((w, h))
    t0=time.perf_counter(); yunet.detect(bench_img_bgr); data.append({"Model":"YuNet","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    # YOLOv8
    t0=time.perf_counter(); yolo_detect(bench_img_rgb,verbose=False); data.append({"Model":"YOLO Detect","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    t0=time.perf_counter(); yolo_pose(bench_img_rgb,verbose=False); data.append({"Model":"YOLO Pose","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    t0=time.perf_counter(); yolo_cls(bench_img_rgb,verbose=False); data.append({"Model":"YOLO Class","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    # YOLO11
    if yolo11_detect is not None:
        t0=time.perf_counter(); yolo11_detect(bench_img_rgb,verbose=False); data.append({"Model":"YOLO11 Detect","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    if yolo11_pose is not None:
        t0=time.perf_counter(); yolo11_pose(bench_img_rgb,verbose=False); data.append({"Model":"YOLO11 Pose","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    if yolo11_cls is not None:
        t0=time.perf_counter(); yolo11_cls(bench_img_rgb,verbose=False); data.append({"Model":"YOLO11 Class","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    # Added YOLO smart preprocess pipelines
    if yolo_seg_v8 is not None:
        t0=time.perf_counter(); _ = yolo_smart_preprocess(yolo_seg_v8, bench_img_rgb, img_size=YOLO_SMART_IMG_SIZE); data.append({"Model":"YOLOv8-Seg SmartCrop","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})
    if yolo_seg_11 is not None:
        t0=time.perf_counter(); _ = yolo_smart_preprocess(yolo_seg_11, bench_img_rgb, img_size=YOLO_SMART_IMG_SIZE); data.append({"Model":"YOLO11-Seg SmartCrop","Run":i,"Latency_ms":(time.perf_counter()-t0)*1000})

    # Segmentation (All Variants)
    if DEVICE=="cuda": torch.cuda.synchronize()
    for name, model in SEG_MODELS.items():
        t0=time.perf_counter()
        with torch.no_grad():
            _=model(torch_input)["out"]
        if DEVICE=="cuda": torch.cuda.synchronize()
        data.append({"Model":name, "Run":i, "Latency_ms":(time.perf_counter()-t0)*1000})

df = pd.DataFrame(data)

summary = df.groupby('Model')['Latency_ms'].agg(['mean','median','std','min','max']).sort_values('mean')
summary['FPS'] = 1000.0 / summary['mean']
summary['P95'] = df.groupby('Model')['Latency_ms'].quantile(0.95)
summary['P99'] = df.groupby('Model')['Latency_ms'].quantile(0.99)
summary = summary.round(2)

MODEL_ORDER = list(summary.index)
MODEL_COLORS = build_color_map(MODEL_ORDER)

print("\n" + "="*100)
print(f"{'FULL MODEL PERFORMANCE REPORT':^100}")
print("="*100)
print(summary[['mean','FPS','std','median','P95','P99']].to_string())
print("="*100 + "\n")

# ---------------- Existing plots (kept) ----------------
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

fig = plt.figure(figsize=(24, 22))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.2])

ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x="FPS", y=summary.index, data=summary, palette=MODEL_COLORS, ax=ax1, orient='h', order=MODEL_ORDER)
ax1.set_title("Throughput (FPS) - Higher is Better", fontweight='bold')
ax1.bar_label(ax1.containers[0], fmt='%.1f')

ax2 = fig.add_subplot(gs[0, 1])
hm_data = summary[['mean', 'median', 'P95', 'std']].copy()
sns.heatmap(hm_data, annot=True, fmt=".1f", cmap="YlGnBu", cbar=False, ax=ax2)
ax2.set_title("Latency Scorecard (ms)", fontweight='bold')

ax3 = fig.add_subplot(gs[1, :])
sns.boxplot(x="Model", y="Latency_ms", data=df, order=MODEL_ORDER, palette=MODEL_COLORS, ax=ax3, showfliers=False)
sns.stripplot(x="Model", y="Latency_ms", data=df, order=MODEL_ORDER, color=".2", size=3, alpha=0.5, ax=ax3)
ax3.set_yscale("log")
ax3.set_title("Latency Distribution (Log Scale)", fontweight='bold')
ax3.tick_params(axis='x', rotation=30)

ax4 = fig.add_subplot(gs[2, :])
for m in MODEL_ORDER:
    sub = df[df["Model"] == m].sort_values("Run")
    ax4.plot(sub["Run"].values, sub["Latency_ms"].values, label=m, color=MODEL_COLORS[m], marker="o", linewidth=2)

ax4.set_title("Inference Stability (Run-by-Run)", fontweight='bold')
ax4.set_ylabel("Latency (ms)")
ax4.legend(bbox_to_anchor=(1.0, 1.0), loc='upper left', ncol=1, frameon=True)

plt.tight_layout()
plt.show()

# ---------------- Added plots (with consistent per-model colors) ----------------

# A) Per-model histograms (FacetGrid)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

def _facet_hist(data, **kws):
    m = data["Model"].iloc[0]
    sns.histplot(data=data, x="Latency_ms", bins=25, kde=False, color=MODEL_COLORS.get(m, "steelblue"))

g = sns.FacetGrid(df, col="Model", col_wrap=4, height=3, sharex=False, sharey=False, col_order=MODEL_ORDER)
g.map_dataframe(_facet_hist)
g.set_titles("{col_name}")
g.fig.suptitle("Per-Model Latency Histograms", y=1.05, fontweight="bold")
plt.tight_layout()
plt.show()

# B) KDE overlay
plt.figure(figsize=(14, 7))
for m in MODEL_ORDER:
    sns.kdeplot(df[df["Model"] == m]["Latency_ms"], label=m, fill=True, alpha=0.22, color=MODEL_COLORS[m])
plt.title("Latency Distribution (KDE Overlay)", fontweight="bold")
plt.xlabel("Latency (ms)")
plt.ylabel("Density")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# C) Violin plot (log scale)
plt.figure(figsize=(14, 7))
sns.violinplot(data=df, x="Model", y="Latency_ms", inner="quartile", order=MODEL_ORDER, palette=MODEL_COLORS, cut=0)
plt.yscale("log")
plt.title("Latency Distribution (Violin Plot, Log Scale)", fontweight="bold")
plt.ylabel("Latency (ms)")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

# D) ECDF overlay
plt.figure(figsize=(14, 7))
for m in MODEL_ORDER:
    vals = df[df["Model"] == m]["Latency_ms"].values
    ecdf = ECDF(vals)
    plt.plot(ecdf.x, ecdf.y, label=m, color=MODEL_COLORS[m], linewidth=2)
plt.xlabel("Latency (ms)")
plt.ylabel("Fraction of Runs ≤ x")
plt.title("Empirical CDF of Latency", fontweight="bold")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# E) Percentiles bar chart (P50/P95/P99)
percentile_df = pd.DataFrame({
    "P50": df.groupby("Model")["Latency_ms"].median(),
    "P95": df.groupby("Model")["Latency_ms"].quantile(0.95),
    "P99": df.groupby("Model")["Latency_ms"].quantile(0.99),
}).loc[MODEL_ORDER].reset_index()

plt.figure(figsize=(14, 7))
x = np.arange(len(MODEL_ORDER))
bar_w = 0.26
plt.bar(x - bar_w, percentile_df["P50"].values, width=bar_w, label="P50")
plt.bar(x,         percentile_df["P95"].values, width=bar_w, label="P95")
plt.bar(x + bar_w, percentile_df["P99"].values, width=bar_w, label="P99")
plt.xticks(x, MODEL_ORDER, rotation=30)
plt.title("Latency Percentiles (P50 / P95 / P99)", fontweight="bold")
plt.ylabel("Latency (ms)")
plt.grid(axis="y", alpha=0.4)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# F) Latency vs Throughput (FPS)
df["Throughput_FPS"] = 1000.0 / df["Latency_ms"]
plt.figure(figsize=(14, 7))
for m in MODEL_ORDER:
    tmp = df[df["Model"] == m]
    plt.scatter(tmp["Latency_ms"], tmp["Throughput_FPS"], s=60, alpha=0.65, label=m, color=MODEL_COLORS[m])
plt.xlabel("Latency (ms)")
plt.ylabel("Throughput (FPS)")
plt.title("Latency vs Throughput Trade-off", fontweight="bold")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# G) Correlation heatmap between metrics
corr_data = summary[["mean", "median", "std", "P95", "FPS"]]
plt.figure(figsize=(8, 6))
sns.heatmap(corr_data.corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Between Performance Metrics", fontweight="bold")
plt.tight_layout()
plt.show()

# H) Stability split into two batches (rolling mean)
unique_models = MODEL_ORDER
split_idx = 7
batch_1 = unique_models[:split_idx]
batch_2 = unique_models[split_idx:]

batches = [
    (batch_1, "Latency Stability - Batch 1"),
    (batch_2, "Latency Stability - Batch 2"),
]

for models_in_batch, title in batches:
    plt.figure(figsize=(14, 7))
    for m in models_in_batch:
        tmp = df[df["Model"] == m].sort_values("Run")
        plt.plot(
            tmp["Run"],
            tmp["Latency_ms"].rolling(window=5, min_periods=1).mean(),
            label=m,
            linewidth=2.5,
            color=MODEL_COLORS[m],
            marker='o',
            markersize=5
        )
    plt.xlabel("Run Index", fontweight='bold')
    plt.ylabel("Latency (ms)", fontweight='bold')
    plt.title(title, fontweight="bold", fontsize=14)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.grid(alpha=0.4, linestyle='--')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================
# TEST ALL ORIGINAL MODELS ON 10 RANDOM IMAGES
# ============================================

NUM_TEST_IMAGES = 10

all_images = [
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

samples = random.sample(all_images, min(NUM_TEST_IMAGES, len(all_images)))

for idx, path in enumerate(samples, 1):
    print(f"\n[{idx}/{len(samples)}] {os.path.basename(path)}")

    img_bgr = cv2.imread(path)
    if img_bgr is None:
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    results = {"Original": img_rgb}

    # -------- MediaPipe --------
    mp_img = mp.Image(mp.ImageFormat.SRGB, img_rgb)
    lm_out = face_landmarker.detect(mp_img)
    mask = np.zeros((h, w), np.uint8)
    if lm_out.face_landmarks:
        pts = np.array([[int(l.x*w), int(l.y*h)] for l in lm_out.face_landmarks[0]])
        cv2.fillConvexPoly(mask, cv2.convexHull(pts), 255)
    results["MediaPipe"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- SSD DNN --------
    blob = cv2.dnn.blobFromImage(img_bgr, 1.0, (300,300), (104,177,123))
    dnn_net.setInput(blob)
    det = dnn_net.forward()
    mask = np.zeros((h, w), np.uint8)
    if det.shape[2] > 0:
        for i in range(det.shape[2]):
            if det[0,0,i,2] > 0.5:
                x1,y1,x2,y2 = (det[0,0,i,3:7]*[w,h,w,h]).astype(int)
                padw = int((x2-x1)*DNN_PADDING_RATIO)
                padh = int((y2-y1)*DNN_PADDING_RATIO)
                mask[max(0,y1-padh):min(h,y2+padh), max(0,x1-padw):min(w,x2+padw)] = 255
                break
    results["DNN SSD"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- Haar --------
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = haar.detectMultiScale(gray, 1.1, 5)
    mask = np.zeros((h, w), np.uint8)
    if len(faces):
        x,y,fw,fh = max(faces, key=lambda b: b[2]*b[3])
        mask[y:y+fh, x:x+fw] = 255
    results["Haar"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- YuNet --------
    yunet.setInputSize((w, h))
    _, faces = yunet.detect(img_bgr)
    mask = np.zeros((h, w), np.uint8)
    if faces is not None and len(faces):
        x,y,bw,bh = faces[0][:4].astype(int)
        mask[y:y+bh, x:x+bw] = 255
    results["YuNet"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- YOLOv8 Detect --------
    r = yolo_detect(img_rgb, conf=0.4, verbose=False)[0]
    mask = np.zeros((h, w), np.uint8)
    if r.boxes and len(r.boxes):
        x1,y1,x2,y2 = r.boxes.xyxy[0].cpu().numpy().astype(int)
        mask[y1:y2, x1:x2] = 255
    results["YOLO Detect"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- YOLOv8 Pose --------
    r = yolo_pose(img_rgb, conf=0.4, verbose=False)[0]
    mask = np.zeros((h, w), np.uint8)
    if r.keypoints is not None and len(r.keypoints.xy):
        kps = r.keypoints.xy[0].cpu().numpy().astype(int)
        if len(kps) >= 3:
            cv2.fillConvexPoly(mask, cv2.convexHull(kps), 255)
    results["YOLO Pose"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- YOLO11 Detect --------
    if yolo11_detect is not None:
        r = yolo11_detect(img_rgb, conf=0.4, verbose=False)[0]
        mask = np.zeros((h, w), np.uint8)
        if r.boxes and len(r.boxes):
            x1,y1,x2,y2 = r.boxes.xyxy[0].cpu().numpy().astype(int)
            mask[y1:y2, x1:x2] = 255
        results["YOLO11 Detect"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- YOLO11 Pose --------
    if yolo11_pose is not None:
        r = yolo11_pose(img_rgb, conf=0.4, verbose=False)[0]
        mask = np.zeros((h, w), np.uint8)
        if r.keypoints is not None and len(r.keypoints.xy):
            kps = r.keypoints.xy[0].cpu().numpy().astype(int)
            if len(kps) >= 3:
                cv2.fillConvexPoly(mask, cv2.convexHull(kps), 255)
        results["YOLO11 Pose"] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- YOLOv8 Seg Smart --------
    if yolo_seg_v8 is not None:
        _, meta = yolo_smart_preprocess(yolo_seg_v8, img_rgb, YOLO_SMART_IMG_SIZE)
        results["YOLOv8-Seg Smart"] = meta["isolated_rgb"]

    # -------- YOLO11 Seg Smart --------
    if yolo_seg_11 is not None:
        _, meta = yolo_smart_preprocess(yolo_seg_11, img_rgb, YOLO_SMART_IMG_SIZE)
        results["YOLO11-Seg Smart"] = meta["isolated_rgb"]

    # -------- Torchvision Seg Models --------
    inp = torch_preprocess(img_rgb).unsqueeze(0).to(DEVICE)
    for name, model in SEG_MODELS.items():
        with torch.no_grad():
            out = model(inp)["out"]
        pred = out[0].argmax(0).cpu().numpy()
        mask = cv2.resize(((pred == 15).astype(np.uint8)*255), (w,h))
        results[name] = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # -------- Display --------
    cols = 4
    rows = (len(results)+cols-1)//cols
    plt.figure(figsize=(24, 6*rows))
    for i,(k,v) in enumerate(results.items(),1):
        plt.subplot(rows, cols, i)
        plt.title(k)
        plt.imshow(v)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


# Face-Centric Image Preprocessing Benchmark  
**Goal:** Black out everything except the human face (side-profile heavy) so downstream models learn from facial geometry, not background.



## 1. Experimental Setup (Summary)

- **Input:** Side-facing face images (not frontal)
- **Objective:** Clean, tight face masks for preprocessing (not detection/classification)
- **Runtime:** CUDA-enabled environment (T4)
- **Evaluation axes:**
  - Latency (mean, P95, P99)
  - Visual mask quality (leakage, face cutting, edge cleanliness)
  - Robustness on profile faces
  - Suitability for mobile deployment


## 2. Models Evaluated

### Detection / Geometry
- MediaPipe FaceLandmarker
- DNN SSD (OpenCV)
- Haar Cascade
- YuNet
- YOLOv8 (Detect, Pose, Class)
- YOLO11 (Detect, Pose, Class)

### Segmentation
- YOLOv8 Seg + SmartCrop
- YOLO11 Seg + SmartCrop
- LRASPP-MobileNetV3
- DeepLabV3-MobileNetV3
- DeepLabV3-ResNet50
- DeepLabV3-ResNet101
- FCN-ResNet50
- FCN-ResNet101

Total tested models: **18**



## 3. Quantitative Performance Results

| Model | Mean (ms) | FPS | P95 (ms) | P99 (ms) |
|-----|----------:|----:|---------:|---------:|
| YOLO Pose | 11.11 | 90.0 | 15.70 | 18.55 |
| YOLO Detect | 11.77 | 85.0 | 16.11 | 19.69 |
| YOLO11 Detect | 12.31 | 81.2 | 17.57 | 20.20 |
| YOLO11 Pose | 13.17 | 75.9 | 19.11 | 21.43 |
| MediaPipe | 16.98 | 58.9 | 24.07 | 26.19 |
| LRASPP-MobileNetV3 | 30.54 | 32.7 | 31.14 | 31.60 |
| YOLOv8-Seg SmartCrop | 32.32 | 30.9 | 46.62 | 50.00 |
| YOLO11-Seg SmartCrop | 35.92 | 27.8 | 49.47 | 53.72 |
| DNN SSD | 51.00 | 19.6 | 71.73 | 76.64 |
| DeepLabV3-MobileNetV3 | 72.86 | 13.7 | 75.67 | 76.73 |
| YuNet | 159.08 | 6.3 | 216.85 | 235.93 |
| DeepLabV3-ResNet50 | 732.09 | 1.37 | 758.93 | 762.83 |
| DeepLabV3-ResNet101 | 1047.99 | 0.95 | 1082.85 | 1086.47 |
| FCN-ResNet50 | 468.98 | 2.13 | 487.99 | 489.97 |
| FCN-ResNet101 | 731.35 | 1.37 | 755.55 | 765.85 |
| Haar | 950.01 | 1.05 | 1170.10 | 1198.39 |



## 4. Qualitative Mask Quality Findings

### Best Mask Quality (Visual)
1. **DeepLabV3-ResNet101**
2. **DeepLabV3-ResNet50**
3. **YOLOv8 Seg + SmartCrop**

- Clean boundaries
- Minimal background leakage
- Robust on side profiles

### Worst Mask Quality
- FCN-ResNet50
- DeepLabV3-MobileNetV3
- LRASPP-MobileNetV3

Common issues:
- Over-smoothing
- Background leakage
- Face cutting (chin, hairline)

### Notable Observations
- **YOLO models** provide the cleanest *outline geometry*
- **DNN SSD** detects faces most reliably but is box-only (no segmentation)
- **YOLOv8 Seg SmartCrop** balances segmentation + speed reasonably well



## 5. Mobile Feasibility Analysis

### Not Mobile-Suitable (Runtime)
- DeepLabV3-ResNet50 / 101
- FCN variants
- Haar

These are **too slow** for on-device inference.

### Mobile-Capable
- YOLOv8 Detect / Pose
- YOLO11 Detect / Pose
- MediaPipe
- YOLOv8 Seg SmartCrop (borderline but acceptable)



## 6. Recommended Hybrid Architecture

### Runtime (Mobile)
**Stage 1 — Face Localization**
- DNN SSD (fast, highly reliable)
- Expand bounding box (≈30–60%) to include full face + profile region

**Stage 2 — Segmentation**
- YOLOv8 Segmentation on expanded ROI
- SmartCrop to normalize scale and framing

**Stage 3 — Cleanup**
- Morphological closing
- Largest connected component
- Optional edge smoothing

### Offline / Training-Time
- Use **DeepLabV3-ResNet50/101** to generate high-quality pseudo-GT masks
- Train a lightweight student model (MobileNetV3 / UNet-lite) for face-only segmentation



## 7. Final Conclusion

- **Best quality masks:** DeepLabV3 (ResNet50/101) — *offline only*
- **Best speed + quality tradeoff:** YOLOv8 Seg + SmartCrop
- **Best hybrid solution:** SSD (expanded ROI) → YOLO Seg → cleanup
- **Best long-term mobile path:** Distilled lightweight segmenter trained from DeepLab pseudo-GT



In [ ]:
import os
import cv2
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image

# ---------------- Config ----------------
IMAGE_DIR = "/content/drive/MyDrive/InnerGize_Face/data/test"
NUM_VISUAL_SAMPLES = 5
BENCHMARK_RUNS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DNN_CONF_THR = 0.5
DNN_PAD_RATIO = 0.40            # expand SSD box by this fraction
YOLO_SEG_CONF = 0.25
SMART_IMG_SIZE = 224            # for YOLO smart-crop baseline

print(f"Device: {DEVICE}")

# ---------------- Fresh downloads (SSD prototxt + weights) ----------------
!rm -f deploy.prototxt res10_300x300_ssd_iter_140000.caffemodel
!wget -q -O deploy.prototxt https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
!wget -q -O res10_300x300_ssd_iter_140000.caffemodel https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel

# ---------------- Load models ----------------
# A) SSD face detector (OpenCV DNN)
dnn_net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "res10_300x300_ssd_iter_140000.caffemodel")

# B) YOLOv8 segmentation (for hybrid + smart-crop baseline)
yolo_seg = YOLO("yolov8n-seg.pt")

# C) DeepLabV3 ResNet50/101 (top quality baselines)
def load_seg(model_fn, weights):
    return model_fn(weights=weights).to(DEVICE).eval()

deeplab_r50  = load_seg(models.segmentation.deeplabv3_resnet50,  models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT)
deeplab_r101 = load_seg(models.segmentation.deeplabv3_resnet101, models.segmentation.DeepLabV3_ResNet101_Weights.DEFAULT)

torch_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------- Utils ----------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def largest_cc_mask(mask01: np.ndarray) -> np.ndarray:
    # mask01: 0/1
    m = (mask01.astype(np.uint8) * 255)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return mask01
    best = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == best).astype(np.uint8)

def cleanup_mask(mask01: np.ndarray) -> np.ndarray:
    # Close holes + keep largest component
    m = (mask01.astype(np.uint8) * 255)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=1)
    m01 = (m > 127).astype(np.uint8)
    m01 = largest_cc_mask(m01)
    return m01

def apply_black_bg(img_rgb: np.ndarray, mask01: np.ndarray) -> np.ndarray:
    mask3 = np.repeat(mask01[:, :, None], 3, axis=2)
    out = np.where(mask3 > 0, img_rgb, 0).astype(np.uint8)
    return out

# ---------------- Baseline 1/2: DeepLabV3 ResNet50/101 ----------------
def run_deeplab_person_mask(model, img_rgb: np.ndarray) -> np.ndarray:
    h, w = img_rgb.shape[:2]
    x = torch_preprocess(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        if DEVICE == "cuda": torch.cuda.synchronize()
        out = model(x)["out"]
        if DEVICE == "cuda": torch.cuda.synchronize()
    pred = out[0].argmax(0).detach().cpu().numpy()   # VOC/COCO style; your earlier used class==15
    mask01 = (pred == 15).astype(np.uint8)
    mask01 = cleanup_mask(mask01)
    return mask01

# ---------------- Baseline 3: YOLOv8-Seg SmartCrop (your style) ----------------
def yolo_smart_preprocess(yolo_seg_model, img_rgb: np.ndarray, img_size=224):
    h, w = img_rgb.shape[:2]
    person_mask = np.zeros((h, w), dtype=np.uint8)

    results = yolo_seg_model(cv2.resize(img_rgb, (640, 640)), conf=YOLO_SEG_CONF, verbose=False)
    if results and results[0].masks is not None and results[0].masks.data is not None and len(results[0].masks.data) > 0:
        areas = results[0].boxes.xywh[:, 2] * results[0].boxes.xywh[:, 3]
        idx = int(torch.argmax(areas).item())
        m = results[0].masks.data[idx].detach().cpu().numpy()
        person_mask = (cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
    else:
        cv2.rectangle(person_mask, (w//4, h//4), (3*w//4, 3*h//4), 1, -1)

    isolated = np.where(np.repeat(person_mask[:, :, None], 3, axis=2) > 0, img_rgb, 0).astype(np.uint8)
    gray_img = cv2.cvtColor(isolated, cv2.COLOR_RGB2GRAY)

    y_head = int(np.argmax(np.any(person_mask, axis=1))) if np.any(person_mask) else 0
    roi_h = max(h // 3, 300)
    dim = max(int(roi_h * 1.8), 300)

    denom = (np.sum(person_mask) + 1e-6)
    center_x = int(np.dot(np.arange(w), np.sum(person_mask, axis=0)) / denom)

    half = dim // 2
    x1 = center_x - half
    y1 = y_head + (roi_h // 2) - half

    p_l, p_t = max(0, -x1), max(0, -y1)
    p_r, p_b = max(0, (x1 + dim) - w), max(0, (y1 + dim) - h)

    padded_gray = cv2.copyMakeBorder(gray_img, p_t, p_b, p_l, p_r, cv2.BORDER_CONSTANT, value=0)
    cx1, cy1 = x1 + p_l, y1 + p_t

    crop_gray = padded_gray[cy1:cy1+dim, cx1:cx1+dim]
    crop_gray = cv2.resize(crop_gray, (img_size, img_size), interpolation=cv2.INTER_AREA)

    # Return the isolated RGB (what you visually inspect)
    return isolated, person_mask

# ---------------- Final candidate: Hybrid SSD ROI -> YOLOv8-seg -> cleanup ----------------
def ssd_face_box(img_bgr: np.ndarray):
    h, w = img_bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(img_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0))
    dnn_net.setInput(blob)
    det = dnn_net.forward()

    best_i, best_c = -1, -1.0
    for i in range(det.shape[2]):
        c = float(det[0, 0, i, 2])
        if c > best_c:
            best_c, best_i = c, i

    if best_i < 0 or best_c < DNN_CONF_THR:
        return None

    box = det[0, 0, best_i, 3:7] * np.array([w, h, w, h])
    x1, y1, x2, y2 = box.astype(int)
    x1, y1, x2, y2 = clamp(x1, 0, w-1), clamp(y1, 0, h-1), clamp(x2, 0, w-1), clamp(y2, 0, h-1)
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1, y1, x2, y2)

def expand_box(box, w, h, pad_ratio=0.4):
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    pad_w, pad_h = int(bw * pad_ratio), int(bh * pad_ratio)
    ex1 = clamp(x1 - pad_w, 0, w-1)
    ey1 = clamp(y1 - pad_h, 0, h-1)
    ex2 = clamp(x2 + pad_w, 0, w-1)
    ey2 = clamp(y2 + pad_h, 0, h-1)
    if ex2 <= ex1 or ey2 <= ey1:
        return (x1, y1, x2, y2)
    return (ex1, ey1, ex2, ey2)

def run_hybrid_ssd_yoloseg(img_bgr: np.ndarray) -> np.ndarray:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    box = ssd_face_box(img_bgr)
    if box is None:
        # Hard fallback: run YOLO seg on full image
        results = yolo_seg(cv2.resize(img_rgb, (640, 640)), conf=YOLO_SEG_CONF, verbose=False)[0]
        mask01 = np.zeros((h, w), dtype=np.uint8)
        if results.masks is not None and len(results.masks.data) > 0:
            areas = results.boxes.xywh[:, 2] * results.boxes.xywh[:, 3]
            idx = int(torch.argmax(areas).item())
            m = results.masks.data[idx].detach().cpu().numpy()
            mask01 = (cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
        mask01 = cleanup_mask(mask01)
        return mask01

    ex1, ey1, ex2, ey2 = expand_box(box, w, h, DNN_PAD_RATIO)
    roi = img_rgb[ey1:ey2, ex1:ex2]
    rh, rw = roi.shape[:2]

    # YOLO seg on ROI
    res = yolo_seg(cv2.resize(roi, (640, 640)), conf=YOLO_SEG_CONF, verbose=False)[0]
    roi_mask01 = np.zeros((rh, rw), dtype=np.uint8)
    if res.masks is not None and len(res.masks.data) > 0:
        areas = res.boxes.xywh[:, 2] * res.boxes.xywh[:, 3]
        idx = int(torch.argmax(areas).item())
        m = res.masks.data[idx].detach().cpu().numpy()
        roi_mask01 = (cv2.resize(m, (rw, rh), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)

    roi_mask01 = cleanup_mask(roi_mask01)

    full_mask01 = np.zeros((h, w), dtype=np.uint8)
    full_mask01[ey1:ey2, ex1:ex2] = roi_mask01
    full_mask01 = cleanup_mask(full_mask01)
    return full_mask01

# ---------------- Load images ----------------
all_images = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
if not all_images:
    raise FileNotFoundError(f"No images found in {IMAGE_DIR}")

samples = random.sample(all_images, min(NUM_VISUAL_SAMPLES, len(all_images)))

# ---------------- Visual compare (Final vs Top-3) ----------------
print("\n>>> VISUAL COMPARISON: Final Hybrid vs Top-3")
for path in samples:
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Final candidate
    m_hybrid = run_hybrid_ssd_yoloseg(img_bgr)
    out_hybrid = apply_black_bg(img_rgb, m_hybrid)

    # Top-3 baselines you reported
    m_r101 = run_deeplab_person_mask(deeplab_r101, img_rgb)
    out_r101 = apply_black_bg(img_rgb, m_r101)

    m_r50 = run_deeplab_person_mask(deeplab_r50, img_rgb)
    out_r50 = apply_black_bg(img_rgb, m_r50)

    out_smart, _ = yolo_smart_preprocess(yolo_seg, img_rgb, img_size=SMART_IMG_SIZE)

    views = {
        "Original": img_rgb,
        "FINAL Hybrid (SSD->YOLOseg)": out_hybrid,
        "DeepLabV3-ResNet101": out_r101,
        "DeepLabV3-ResNet50": out_r50,
        "YOLOv8-Seg SmartCrop": out_smart,
    }

    cols = 3
    rows = (len(views) + cols - 1) // cols
    plt.figure(figsize=(18, 6 * rows))
    for i, (k, v) in enumerate(views.items(), 1):
        plt.subplot(rows, cols, i)
        plt.title(k, fontweight="bold")
        plt.imshow(v)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# ---------------- Benchmark (Final vs Top-3) ----------------
print("\n>>> BENCHMARK: Final Hybrid vs Top-3")

bench_path = samples[0]
bench_bgr = cv2.imread(bench_path)
if bench_bgr is None:
    raise ValueError("Benchmark image could not be read.")
bench_rgb = cv2.cvtColor(bench_bgr, cv2.COLOR_BGR2RGB)

def time_it(name, fn, runs=BENCHMARK_RUNS):
    times = []
    # warmup
    for _ in range(3):
        _ = fn()
    for i in range(runs):
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = fn()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return pd.DataFrame({"Model": name, "Run": np.arange(runs), "Latency_ms": times})

dfs = []

dfs.append(time_it("FINAL Hybrid (SSD->YOLOseg)", lambda: run_hybrid_ssd_yoloseg(bench_bgr)))

dfs.append(time_it("DeepLabV3-ResNet101", lambda: run_deeplab_person_mask(deeplab_r101, bench_rgb)))
dfs.append(time_it("DeepLabV3-ResNet50",  lambda: run_deeplab_person_mask(deeplab_r50,  bench_rgb)))

dfs.append(time_it("YOLOv8-Seg SmartCrop", lambda: yolo_smart_preprocess(yolo_seg, bench_rgb, img_size=SMART_IMG_SIZE)))

df = pd.concat(dfs, ignore_index=True)

summary = df.groupby("Model")["Latency_ms"].agg(["mean", "median", "std", "min", "max"]).sort_values("mean")
summary["FPS"] = 1000.0 / summary["mean"]
summary["P95"] = df.groupby("Model")["Latency_ms"].quantile(0.95)
summary["P99"] = df.groupby("Model")["Latency_ms"].quantile(0.99)
summary = summary.round(2)

print("\n" + "="*90)
print(f"{'FINAL vs TOP-3 PERFORMANCE REPORT':^90}")
print("="*90)
print(summary[["mean", "FPS", "std", "median", "P95", "P99"]].to_string())
print("="*90 + "\n")

# Consistent colors across plots
MODEL_ORDER = list(summary.index)
palette = sns.color_palette("tab10", n_colors=len(MODEL_ORDER))
MODEL_COLORS = {m: palette[i] for i, m in enumerate(MODEL_ORDER)}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

# FPS bar
plt.figure(figsize=(10, 5))
sns.barplot(x=summary["FPS"], y=MODEL_ORDER, palette=MODEL_COLORS, orient="h")
plt.title("Throughput (FPS) - Final vs Top-3", fontweight="bold")
plt.xlabel("FPS")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Box plot (log scale)
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="Model", y="Latency_ms", order=MODEL_ORDER, palette=MODEL_COLORS, showfliers=False)
plt.yscale("log")
plt.title("Latency Distribution (Log Scale)", fontweight="bold")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Stability plot
plt.figure(figsize=(12, 5))
for m in MODEL_ORDER:
    tmp = df[df["Model"] == m].sort_values("Run")
    plt.plot(tmp["Run"], tmp["Latency_ms"], label=m, color=MODEL_COLORS[m], linewidth=2)
plt.title("Run-by-Run Latency Stability", fontweight="bold")
plt.xlabel("Run")
plt.ylabel("Latency (ms)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import os
import cv2
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from ultralytics import YOLO
from torchvision import models, transforms

# ---------------- Config ----------------
IMAGE_DIR = "/content/drive/MyDrive/InnerGize_Face/data/test"
NUM_VISUAL_SAMPLES = 5
BENCHMARK_RUNS = 120
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

YOLO_DET_CONF = 0.25
YOLO_SEG_CONF = 0.25
ROI_EXPAND_RATIO = 0.55
SMARTCROP_IMG_SIZE = 224

print(f"Device: {DEVICE}")

# ---------------- Models ----------------
yolo_det = YOLO("yolov8n.pt")
yolo_seg = YOLO("yolov8n-seg.pt")

def load_seg(model_fn, weights):
    return model_fn(weights=weights).to(DEVICE).eval()

deeplab_r50  = load_seg(models.segmentation.deeplabv3_resnet50,  models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT)
deeplab_r101 = load_seg(models.segmentation.deeplabv3_resnet101, models.segmentation.DeepLabV3_ResNet101_Weights.DEFAULT)

torch_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------- Utils ----------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def largest_cc_mask(mask01: np.ndarray) -> np.ndarray:
    m = (mask01.astype(np.uint8) * 255)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return mask01
    best = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == best).astype(np.uint8)

def cleanup_mask(mask01: np.ndarray) -> np.ndarray:
    m = (mask01.astype(np.uint8) * 255)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=1)
    m01 = (m > 127).astype(np.uint8)
    m01 = largest_cc_mask(m01)
    return m01

def apply_black_bg(img_rgb: np.ndarray, mask01: np.ndarray) -> np.ndarray:
    mask3 = np.repeat(mask01[:, :, None], 3, axis=2)
    return np.where(mask3 > 0, img_rgb, 0).astype(np.uint8)

def pick_adaptive_imgsz(roi_w, roi_h, full_w, full_h):
    frac = (roi_w * roi_h) / float(full_w * full_h + 1e-6)
    m = max(roi_w, roi_h)
    if frac <= 0.10 or m <= 220:
        return 320
    if frac <= 0.25 or m <= 420:
        return 416
    return 640

def yolo_det_box(img_rgb: np.ndarray):
    r = yolo_det(img_rgb, conf=YOLO_DET_CONF, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None
    boxes = r.boxes.xyxy.detach().cpu().numpy()
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    i = int(np.argmax(areas))
    x1, y1, x2, y2 = boxes[i].astype(int)
    return (x1, y1, x2, y2)

def expand_box(box, w, h, ratio=0.55):
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    pad_w, pad_h = int(bw * ratio), int(bh * ratio)
    ex1 = clamp(x1 - pad_w, 0, w - 1)
    ey1 = clamp(y1 - pad_h, 0, h - 1)
    ex2 = clamp(x2 + pad_w, 0, w - 1)
    ey2 = clamp(y2 + pad_h, 0, h - 1)
    if ex2 <= ex1 or ey2 <= ey1:
        return box
    return (ex1, ey1, ex2, ey2)

def yolo_seg_mask_for_image(img_rgb: np.ndarray, imgsz=640) -> np.ndarray:
    h, w = img_rgb.shape[:2]
    r = yolo_seg(img_rgb, imgsz=imgsz, conf=YOLO_SEG_CONF, verbose=False)[0]
    mask01 = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None and r.masks.data is not None and len(r.masks.data) > 0:
        boxes = r.boxes.xyxy.detach().cpu().numpy()
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        i = int(np.argmax(areas))
        m = r.masks.data[i].detach().cpu().numpy()
        mask01 = (cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
    return cleanup_mask(mask01)

# ---------------- Top-3 baseline masks ----------------
def run_deeplab_person_mask(model, img_rgb: np.ndarray) -> np.ndarray:
    x = torch_preprocess(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        if DEVICE == "cuda": torch.cuda.synchronize()
        out = model(x)["out"]
        if DEVICE == "cuda": torch.cuda.synchronize()
    pred = out[0].argmax(0).detach().cpu().numpy()
    mask01 = (pred == 15).astype(np.uint8)
    return cleanup_mask(mask01)

def yolo_smart_preprocess_isolated(yolo_seg_model, img_rgb: np.ndarray, img_size=224):
    h, w = img_rgb.shape[:2]
    r = yolo_seg_model(cv2.resize(img_rgb, (640, 640)), imgsz=640, conf=YOLO_SEG_CONF, verbose=False)[0]
    person_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None and r.masks.data is not None and len(r.masks.data) > 0:
        boxes = r.boxes.xyxy.detach().cpu().numpy()
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        i = int(np.argmax(areas))
        m = r.masks.data[i].detach().cpu().numpy()
        person_mask = (cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
    else:
        cv2.rectangle(person_mask, (w//4, h//4), (3*w//4, 3*h//4), 1, -1)

    isolated = apply_black_bg(img_rgb, cleanup_mask(person_mask))
    gray_img = cv2.cvtColor(isolated, cv2.COLOR_RGB2GRAY)

    y_head = int(np.argmax(np.any(person_mask, axis=1))) if np.any(person_mask) else 0
    roi_h = max(h // 3, 300)
    dim = max(int(roi_h * 1.8), 300)

    denom = (np.sum(person_mask) + 1e-6)
    center_x = int(np.dot(np.arange(w), np.sum(person_mask, axis=0)) / denom)

    half = dim // 2
    x1 = center_x - half
    y1 = y_head + (roi_h // 2) - half

    p_l, p_t = max(0, -x1), max(0, -y1)
    p_r, p_b = max(0, (x1 + dim) - w), max(0, (y1 + dim) - h)

    padded_gray = cv2.copyMakeBorder(gray_img, p_t, p_b, p_l, p_r, cv2.BORDER_CONSTANT, value=0)
    cx1, cy1 = x1 + p_l, y1 + p_t

    crop_gray = padded_gray[cy1:cy1+dim, cx1:cx1+dim]
    crop_gray = cv2.resize(crop_gray, (img_size, img_size), interpolation=cv2.INTER_AREA)

    return isolated, cleanup_mask(person_mask)

# ---------------- Hybrid that can win ----------------
def run_hybrid_yolo_det_to_yolo_seg_adaptive(img_rgb: np.ndarray) -> np.ndarray:
    h, w = img_rgb.shape[:2]
    box = yolo_det_box(img_rgb)

    if box is None:
        return yolo_seg_mask_for_image(img_rgb, imgsz=640)

    ex1, ey1, ex2, ey2 = expand_box(box, w, h, ROI_EXPAND_RATIO)
    roi = img_rgb[ey1:ey2, ex1:ex2]
    rh, rw = roi.shape[:2]

    imgsz = pick_adaptive_imgsz(rw, rh, w, h)

    roi_mask = yolo_seg_mask_for_image(roi, imgsz=imgsz)

    full_mask = np.zeros((h, w), dtype=np.uint8)
    full_mask[ey1:ey2, ex1:ex2] = roi_mask
    return cleanup_mask(full_mask)

# ---------------- Data loading ----------------
all_images = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
if not all_images:
    raise FileNotFoundError(f"No images found in {IMAGE_DIR}")

samples = random.sample(all_images, min(NUM_VISUAL_SAMPLES, len(all_images)))
bench_path = samples[0]

bench_bgr = cv2.imread(bench_path)
if bench_bgr is None:
    raise ValueError("Benchmark image could not be read")
bench_rgb = cv2.cvtColor(bench_bgr, cv2.COLOR_BGR2RGB)

# ---------------- Visual compare ----------------
print("\n>>> VISUAL COMPARISON: Hybrid vs Top-3")
for path in samples:
    bgr = cv2.imread(path)
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    m_hybrid = run_hybrid_yolo_det_to_yolo_seg_adaptive(rgb)
    out_hybrid = apply_black_bg(rgb, m_hybrid)

    m_r101 = run_deeplab_person_mask(deeplab_r101, rgb)
    out_r101 = apply_black_bg(rgb, m_r101)

    m_r50 = run_deeplab_person_mask(deeplab_r50, rgb)
    out_r50 = apply_black_bg(rgb, m_r50)

    out_smart, m_smart = yolo_smart_preprocess_isolated(yolo_seg, rgb, img_size=SMARTCROP_IMG_SIZE)

    views = {
        "Original": rgb,
        "HYBRID (YOLOdet->ROI->YOLOseg adaptive)": out_hybrid,
        "YOLOv8-Seg SmartCrop": out_smart,
        "DeepLabV3-ResNet50": out_r50,
        "DeepLabV3-ResNet101": out_r101,
    }

    cols = 3
    rows = (len(views) + cols - 1) // cols
    plt.figure(figsize=(18, 6 * rows))
    for i, (k, v) in enumerate(views.items(), 1):
        plt.subplot(rows, cols, i)
        plt.title(k, fontweight="bold")
        plt.imshow(v)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# ---------------- Benchmark ----------------
def time_it(name, fn, runs=BENCHMARK_RUNS):
    times = []
    for _ in range(5):
        _ = fn()
    for i in range(runs):
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = fn()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return pd.DataFrame({"Model": name, "Run": np.arange(runs), "Latency_ms": times})

dfs = []
dfs.append(time_it("HYBRID (YOLOdet->ROI->YOLOseg adaptive)", lambda: run_hybrid_yolo_det_to_yolo_seg_adaptive(bench_rgb)))
dfs.append(time_it("YOLOv8-Seg SmartCrop", lambda: yolo_smart_preprocess_isolated(yolo_seg, bench_rgb, img_size=SMARTCROP_IMG_SIZE)))
dfs.append(time_it("DeepLabV3-ResNet50",  lambda: run_deeplab_person_mask(deeplab_r50,  bench_rgb)))
dfs.append(time_it("DeepLabV3-ResNet101", lambda: run_deeplab_person_mask(deeplab_r101, bench_rgb)))

df = pd.concat(dfs, ignore_index=True)

summary = df.groupby("Model")["Latency_ms"].agg(["mean", "median", "std", "min", "max"]).sort_values("mean")
summary["FPS"] = 1000.0 / summary["mean"]
summary["P95"] = df.groupby("Model")["Latency_ms"].quantile(0.95)
summary["P99"] = df.groupby("Model")["Latency_ms"].quantile(0.99)
summary = summary.round(2)

print("\n" + "="*105)
print(f"{'HYBRID vs TOP-3 PERFORMANCE REPORT':^105}")
print("="*105)
print(summary[["mean", "FPS", "std", "median", "P95", "P99"]].to_string())
print("="*105 + "\n")

# ---------------- Plots (consistent colors) ----------------
MODEL_ORDER = list(summary.index)
palette = sns.color_palette("tab10", n_colors=len(MODEL_ORDER))
MODEL_COLORS = {m: palette[i] for i, m in enumerate(MODEL_ORDER)}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

# FPS
plt.figure(figsize=(10, 5))
sns.barplot(x=summary["FPS"], y=MODEL_ORDER, palette=MODEL_COLORS, orient="h")
plt.title("Throughput (FPS) - Hybrid vs Top-3", fontweight="bold")
plt.xlabel("FPS")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Latency distribution
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="Model", y="Latency_ms", order=MODEL_ORDER, palette=MODEL_COLORS, showfliers=False)
plt.yscale("log")
plt.title("Latency Distribution (Log Scale)", fontweight="bold")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Stability
plt.figure(figsize=(12, 5))
for m in MODEL_ORDER:
    tmp = df[df["Model"] == m].sort_values("Run")
    plt.plot(tmp["Run"], tmp["Latency_ms"], label=m, color=MODEL_COLORS[m], linewidth=2)
plt.title("Run-by-Run Latency Stability", fontweight="bold")
plt.xlabel("Run")
plt.ylabel("Latency (ms)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


# Face-Centric Preprocessing Pipeline – Final Findings & Decision

## Objective
Design a face-only preprocessing pipeline that:
- Blacks out everything except the face (side-profile heavy dataset)
- Produces geometrically meaningful masks
- Is fast, stable, and suitable for mobile deployment
- Is justified by hard latency and quality evidence

## Experimental Scope

### Models Evaluated

Detection / Geometry
- YOLOv8n Detect
- YOLOv8n Pose
- YOLO11 Detect / Pose
- DNN SSD (OpenCV)
- MediaPipe FaceLandmarker

Segmentation
- YOLOv8n-Seg (SmartCrop)
- YOLO11-Seg (SmartCrop)
- DeepLabV3-ResNet50
- DeepLabV3-ResNet101
- DeepLabV3-MobileNetV3
- LRASPP-MobileNetV3
- FCN-ResNet50 / ResNet101

Total models evaluated: 18

## Key Experimental Results (Hard Facts)

### Final Head-to-Head (Relevant Models Only)

| Model | Mean (ms) | FPS | P95 (ms) | P99 (ms) |
|------|----------:|----:|---------:|---------:|
| HYBRID (YOLOdet → ROI → YOLOseg adaptive) | 67.68 | 14.77 | 87.33 | 92.96 |
| YOLOv8-Seg SmartCrop | 78.49 | 12.74 | 108.17 | 118.80 |
| DeepLabV3-ResNet50 | 758.50 | 1.32 | 783.90 | 799.10 |
| DeepLabV3-ResNet101 | 1115.07 | 0.90 | 1133.80 | 1138.99 |

### Measured Improvements of HYBRID vs YOLOv8-Seg SmartCrop
- Mean latency reduced by 13.8%
- P95 latency reduced by 19.3%
- P99 latency reduced by 21.7%
- Throughput increased by 15.9%

These improvements are consistent across runs and materially improve tail latency.

## Visual Quality Findings

Best mask quality (visual inspection)
1. DeepLabV3-ResNet101
2. DeepLabV3-ResNet50
3. HYBRID ≈ YOLOv8-Seg SmartCrop

Critical observation  
Hybrid and YOLOv8-Seg SmartCrop are visually almost indistinguishable:
- No meaningful difference in face coverage
- No additional background leakage
- No increased face cutting on side profiles

This removes visual quality as a deciding factor between the two.

## Why the Hybrid Wins

The decisive factor is adaptive segmentation resolution.

- YOLOv8-Seg SmartCrop always segments at 640×640
- Hybrid dynamically selects 320 / 416 / 640 based on ROI size
- ROI is obtained cheaply using YOLO Detect (~10–12 ms)
- Face-centric images frequently produce small ROIs

This results in:
- Lower average latency
- Much better P95 and P99 behavior
- No observable loss in mask quality

## Final Model Ranking

Runtime (mobile-facing)
1. HYBRID (YOLOv8 Detect → ROI → YOLOv8-Seg adaptive)
2. YOLOv8-Seg SmartCrop
3. DeepLabV3-ResNet50
4. DeepLabV3-ResNet101

Mask quality (offline reference)
1. DeepLabV3-ResNet101
2. DeepLabV3-ResNet50
3. HYBRID ≈ YOLOv8-Seg SmartCrop

## Final Runtime Pipeline (Locked)

YOLOv8n Detect → Expanded ROI → YOLOv8n-Seg (adaptive imgsz) → Morphological cleanup

Locked parameters
- ROI expansion ratio: 0.5–0.6
- Adaptive segmentation resolution:
  - Small ROI → 320
  - Medium ROI → 416
  - Large ROI → 640
- Post-processing:
  - Morphological close
  - Keep largest connected component

These settings are experimentally validated and should not be changed without new data.

## Role of DeepLab Models

DeepLabV3-ResNet50 and DeepLabV3-ResNet101:
- Not suitable for mobile runtime
- Produce the highest-quality masks
- Serve as offline pseudo-ground-truth generators
- Act as teacher models for future lightweight students

They are explicitly excluded from deployment.

## Final Conclusion

The HYBRID pipeline:
- Preserves mask quality
- Beats YOLOv8-Seg SmartCrop on mean latency and tail latency
- Is realistically deployable on mobile hardware

Model selection is complete.  
This pipeline is technically defensible, empirically justified, and ready for engineering integration.

## Optional Next Steps
- Package the hybrid as a standalone preprocessing module
- Benchmark CPU-only performance for mobile realism
- Distill DeepLab supervision into a lightweight student segmenter
- Prepare TFLite / CoreML conversion paths
